# Follow-Up After Univariate Correlations

Reusable diagnostics for validating interesting univariate correlations after an exploratory `run_analysis` pass. The workflow is comparison-agnostic: pick any comparison from the loaded config, select predictors, and inspect scatter plots, predictor redundancy, controlled regressions, and leave-one-dataset-out robustness.

## Edit These Knobs

Set `COMPARISON_NAME` to one of the printed comparison names, or leave it as `None` to use the first comparison in the config. Leave `INTERESTING_PREDICTORS` empty to auto-select the top predictors by adjusted p-value when available, otherwise raw p-value.

In [ ]:
CONFIG_NAME = "config_4.yaml"
DATASETS = None
COMPARISON_NAME = None
INTERESTING_PREDICTORS = []
CONTROL_PREDICTORS = ["log_n", "log_d", "cat_fraction", "missing_fraction"]
TOP_K = 12

## Environment and Imports

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path
from warnings import catch_warnings, simplefilter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
from scipy.stats import spearmanr
from statsmodels.stats.outliers_influence import variance_inflation_factor

cwd = Path.cwd()
if (cwd / "src" / "mfa").exists():
    project_dir = cwd
elif cwd.name == "notebooks" and (cwd.parent / "src" / "mfa").exists():
    project_dir = cwd.parent
elif (cwd / "meta-feature-analysis" / "src" / "mfa").exists():
    project_dir = cwd / "meta-feature-analysis"
else:
    raise RuntimeError("Run this notebook from meta-feature-analysis/ or its notebooks/ directory.")
repo_root = project_dir.parent

path_candidates = [
    project_dir / "src",
    repo_root / "tabarena" / "tabarena",
    repo_root / "tabarena" / "bencheval",
    repo_root / "tabarena" / "tabflow",
]
for path in path_candidates:
    if path.exists():
        sys.path.insert(0, str(path))

from mfa import load_config, run_analysis

sns.set_theme(style="ticks", context="notebook")

## Run or Load Analysis

In [ ]:
config_path = project_dir / "configs" / CONFIG_NAME
config = load_config(config_path)
comparison_names = [comparison.name for comparison in config.comparisons]

print(f"Loaded config: {config_path}")
print("Available comparisons:")
for name in comparison_names:
    print(f"  - {name}")

selected_comparison = COMPARISON_NAME or comparison_names[0]
if selected_comparison not in comparison_names:
    raise ValueError(
        f"COMPARISON_NAME={selected_comparison!r} is not in the config. "
        f"Available comparisons: {comparison_names}"
    )
if COMPARISON_NAME is None:
    print(f"COMPARISON_NAME is None; using first comparison: {selected_comparison}")

result = run_analysis(config, datasets=DATASETS)
analysis = result.analysis_table.copy()

corr_df = pd.DataFrame([vars(row) for row in result.correlation_results])
if result.correction_result is not None and not corr_df.empty:
    adjusted = list(result.correction_result.adjusted_p_values)
    rejected = list(result.correction_result.rejected)
    if len(adjusted) == len(corr_df):
        corr_df["p_value_adj"] = adjusted
        corr_df["rejected"] = rejected
    else:
        correction_df = pd.DataFrame([vars(row) for row in result.correction_result.results])
        correction_df["p_value_adj"] = adjusted
        correction_df["rejected"] = rejected
        key_cols = ["comparison_name", "predictor", "target", "statistic", "p_value", "n_observations"]
        corr_df = corr_df.merge(
            correction_df[key_cols + ["p_value_adj", "rejected"]],
            on=key_cols,
            how="left",
        )

print(f"analysis_table shape: {analysis.shape}")
print(f"correlation rows: {len(corr_df)}")
print(f"selected comparison: {selected_comparison}")
display(analysis.head())

## Correlation Triage Table

In [ ]:
def _available_columns(frame: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in frame.columns]


analysis_comp = analysis[analysis["comparison_name"].eq(selected_comparison)].copy()
corr_comp = corr_df[corr_df["comparison_name"].eq(selected_comparison)].copy()
if corr_comp.empty:
    raise ValueError(f"No correlation results found for comparison {selected_comparison!r}.")

p_sort_col = "p_value_adj" if "p_value_adj" in corr_comp.columns and corr_comp["p_value_adj"].notna().any() else "p_value"
corr_comp = corr_comp.sort_values([p_sort_col, "p_value", "predictor"], na_position="last").reset_index(drop=True)
corr_comp["univariate_rank"] = np.arange(1, len(corr_comp) + 1)

if INTERESTING_PREDICTORS:
    requested_predictors = list(dict.fromkeys(INTERESTING_PREDICTORS))
else:
    requested_predictors = corr_comp["predictor"].head(TOP_K).tolist()

present_predictors = [predictor for predictor in requested_predictors if predictor in analysis_comp.columns]
missing_predictors = [predictor for predictor in requested_predictors if predictor not in analysis_comp.columns]
selected_predictors = [
    predictor
    for predictor in present_predictors
    if pd.api.types.is_numeric_dtype(analysis_comp[predictor])
]
non_numeric_predictors = [predictor for predictor in present_predictors if predictor not in selected_predictors]

triage_cols = _available_columns(
    corr_comp,
    [
        "univariate_rank",
        "predictor",
        "statistic",
        "ci_lower",
        "ci_upper",
        "p_value",
        "p_value_adj",
        "rejected",
        "n_observations",
        "direction_confirmed",
    ],
)

print(f"Sorted by: {p_sort_col}")
display(corr_comp[triage_cols].head(max(TOP_K, len(selected_predictors))))

print("Selected predictors present in analysis:")
for predictor in selected_predictors:
    print(f"  - {predictor}")
if missing_predictors:
    print(f"Missing predictors ignored: {missing_predictors}")
if non_numeric_predictors:
    print(f"Non-numeric predictors ignored: {non_numeric_predictors}")

display(corr_comp[corr_comp["predictor"].isin(selected_predictors)][triage_cols])

## Scatter Diagnostics

In [ ]:
def _format_p_value(value: float) -> str:
    if pd.isna(value):
        return "NA"
    if value < 0.001:
        return f"{value:.1e}"
    return f"{value:.3f}"


if not selected_predictors:
    print("No numeric selected predictors to plot.")
else:
    ncols = min(3, len(selected_predictors))
    nrows = math.ceil(len(selected_predictors) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.2 * nrows), squeeze=False)

    for idx, predictor in enumerate(selected_predictors):
        ax = axes.flat[idx]
        plot_df = analysis_comp[[predictor, "delta_norm"]].replace([np.inf, -np.inf], np.nan).dropna()
        if plot_df.empty or plot_df[predictor].nunique() < 2:
            ax.text(0.5, 0.5, "insufficient variation", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue

        sns.scatterplot(
            data=plot_df,
            x=predictor,
            y="delta_norm",
            s=42,
            alpha=0.78,
            edgecolor="white",
            linewidth=0.4,
            ax=ax,
        )
        ax.axhline(0, color="0.35", linestyle=":", linewidth=1)

        x_values = plot_df[predictor]
        use_log = bool((x_values > 0).all() and (x_values.max() / x_values.min() > 10))
        if use_log:
            ax.set_xscale("log")

        corr_row = corr_comp[corr_comp["predictor"].eq(predictor)]
        if not corr_row.empty:
            corr_row = corr_row.iloc[0]
            p_value = corr_row[p_sort_col] if p_sort_col in corr_row.index else corr_row["p_value"]
            p_label = "adj p" if p_sort_col == "p_value_adj" else "p"
            annotation = (
                f"rho={corr_row['statistic']:.2f}\n"
                f"{p_label}={_format_p_value(p_value)}, n={int(corr_row['n_observations'])}"
            )
            ax.text(
                0.04,
                0.96,
                annotation,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=9,
                bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9, "pad": 4},
            )

        ax.set_xlabel(predictor)
        ax.set_ylabel("delta_norm = group_a_error - group_b_error")

    for idx in range(len(selected_predictors), nrows * ncols):
        axes.flat[idx].set_axis_off()

    fig.suptitle(f"{selected_comparison}: selected univariate relationships", y=1.02)
    fig.tight_layout()

## Predictor Redundancy

In [ ]:
def _unique(values: list[str]) -> list[str]:
    return list(dict.fromkeys(values))


present_controls = [
    predictor
    for predictor in CONTROL_PREDICTORS
    if predictor in analysis_comp.columns and pd.api.types.is_numeric_dtype(analysis_comp[predictor])
]
missing_controls = [predictor for predictor in CONTROL_PREDICTORS if predictor not in present_controls]
if missing_controls:
    print(f"Control predictors unavailable or non-numeric: {missing_controls}")

redundancy_columns = _unique(selected_predictors + present_controls)
usable_redundancy_columns = [
    column
    for column in redundancy_columns
    if analysis_comp[column].replace([np.inf, -np.inf], np.nan).dropna().nunique() > 1
]

high_corr_pairs = pd.DataFrame(columns=["predictor_1", "predictor_2", "rho", "abs_rho"])
if len(usable_redundancy_columns) < 2:
    print("Need at least two non-constant predictors for redundancy diagnostics.")
else:
    redundancy_corr = analysis_comp[usable_redundancy_columns].replace([np.inf, -np.inf], np.nan).corr(
        method="spearman",
        min_periods=3,
    )
    fig_width = max(6, min(14, 0.55 * len(usable_redundancy_columns) + 4))
    fig, ax = plt.subplots(figsize=(fig_width, fig_width * 0.8))
    sns.heatmap(
        redundancy_corr,
        vmin=-1,
        vmax=1,
        cmap="vlag",
        center=0,
        annot=len(usable_redundancy_columns) <= 12,
        fmt=".2f",
        square=True,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title("Spearman redundancy among selected predictors and controls")
    fig.tight_layout()

    pair_rows = []
    for i, left in enumerate(usable_redundancy_columns):
        for right in usable_redundancy_columns[i + 1 :]:
            rho = redundancy_corr.loc[left, right]
            if pd.notna(rho) and abs(rho) >= 0.7:
                pair_rows.append(
                    {
                        "predictor_1": left,
                        "predictor_2": right,
                        "rho": float(rho),
                        "abs_rho": float(abs(rho)),
                    }
                )
    high_corr_pairs = pd.DataFrame(pair_rows).sort_values("abs_rho", ascending=False) if pair_rows else high_corr_pairs

print("High-correlation pairs with abs(rho) >= 0.7:")
display(high_corr_pairs)

## Controlled Regressions

In [ ]:
def _standardize_predictors(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(index=frame.index)
    numeric = frame.astype(float)
    scale = numeric.std(ddof=0).replace(0, np.nan)
    keep = scale[scale.notna()].index.tolist()
    if not keep:
        return pd.DataFrame(index=frame.index)
    return (numeric[keep] - numeric[keep].mean()) / scale[keep]


def _design_matrix(frame: pd.DataFrame, predictors: list[str]) -> pd.DataFrame:
    x_std = _standardize_predictors(frame[predictors]) if predictors else pd.DataFrame(index=frame.index)
    intercept = pd.Series(1.0, index=frame.index, name="const")
    return pd.concat([intercept, x_std], axis=1)


def _fit_ols(frame: pd.DataFrame, predictors: list[str]):
    x = _design_matrix(frame, predictors)
    y = frame["delta_norm"].astype(float)
    if len(frame) <= x.shape[1]:
        return None, x, "not enough residual degrees of freedom"
    try:
        model = sm.OLS(y, x).fit()
    except Exception as exc:
        return None, x, str(exc)
    return model, x, None


def _max_vif(design_matrix: pd.DataFrame) -> float:
    predictor_columns = [column for column in design_matrix.columns if column != "const"]
    if not predictor_columns:
        return np.nan
    values = design_matrix.astype(float).to_numpy()
    vifs = []
    with catch_warnings():
        simplefilter("ignore")
        for idx, column in enumerate(design_matrix.columns):
            if column == "const":
                continue
            try:
                vifs.append(float(variance_inflation_factor(values, idx)))
            except Exception:
                vifs.append(np.nan)
    if any(np.isinf(value) for value in vifs):
        return np.inf
    finite_vifs = [value for value in vifs if np.isfinite(value)]
    return max(finite_vifs) if finite_vifs else np.nan


regression_rows = []
for predictor in selected_predictors:
    control_candidates = [control for control in present_controls if control != predictor]
    model_columns = _unique(["delta_norm", predictor] + control_candidates)
    model_data = analysis_comp[model_columns].apply(pd.to_numeric, errors="coerce")
    model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna()

    if predictor not in model_data.columns or model_data[predictor].nunique() < 2:
        regression_rows.append(
            {
                "predictor": predictor,
                "coefficient": np.nan,
                "p_value": np.nan,
                "n_observations": int(len(model_data)),
                "adj_r_squared": np.nan,
                "delta_adj_r_squared": np.nan,
                "max_vif": np.nan,
                "controls_used": ", ".join(control_candidates),
                "status": "predictor missing or constant after dropping missing rows",
            }
        )
        continue

    active_controls = [control for control in control_candidates if model_data[control].nunique() > 1]
    base_predictors = active_controls
    extended_predictors = active_controls + [predictor]

    base_model, _, base_error = _fit_ols(model_data[["delta_norm"] + base_predictors], base_predictors)
    extended_model, extended_x, extended_error = _fit_ols(
        model_data[["delta_norm"] + extended_predictors],
        extended_predictors,
    )

    if extended_model is None:
        regression_rows.append(
            {
                "predictor": predictor,
                "coefficient": np.nan,
                "p_value": np.nan,
                "n_observations": int(len(model_data)),
                "adj_r_squared": np.nan,
                "delta_adj_r_squared": np.nan,
                "max_vif": np.nan,
                "controls_used": ", ".join(active_controls),
                "status": extended_error,
            }
        )
        continue

    base_adj_r2 = float(base_model.rsquared_adj) if base_model is not None else np.nan
    extended_adj_r2 = float(extended_model.rsquared_adj)
    regression_rows.append(
        {
            "predictor": predictor,
            "coefficient": float(extended_model.params.get(predictor, np.nan)),
            "p_value": float(extended_model.pvalues.get(predictor, np.nan)),
            "n_observations": int(extended_model.nobs),
            "adj_r_squared": extended_adj_r2,
            "delta_adj_r_squared": extended_adj_r2 - base_adj_r2 if pd.notna(base_adj_r2) else np.nan,
            "max_vif": _max_vif(extended_x),
            "controls_used": ", ".join(active_controls),
            "status": "ok" if base_error is None else f"extended ok; base issue: {base_error}",
        }
    )

regression_results = pd.DataFrame(regression_rows)
regression_cols = [
    "predictor",
    "coefficient",
    "p_value",
    "n_observations",
    "adj_r_squared",
    "delta_adj_r_squared",
    "max_vif",
    "controls_used",
    "status",
]
display(regression_results[regression_cols] if not regression_results.empty else regression_results)

## Leave-One-Dataset-Out Robustness

In [ ]:
def _safe_spearman(frame: pd.DataFrame, predictor: str) -> tuple[float, int]:
    subset = frame[[predictor, "delta_norm"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(subset) < 3 or subset[predictor].nunique() < 2 or subset["delta_norm"].nunique() < 2:
        return np.nan, int(len(subset))
    statistic, _ = spearmanr(subset[predictor], subset["delta_norm"])
    return float(statistic), int(len(subset))


lodo_rows = []
lodo_detail_rows = []
dataset_values = sorted(analysis_comp["dataset"].dropna().unique()) if "dataset" in analysis_comp.columns else []

for predictor in selected_predictors:
    full_statistic, full_n = _safe_spearman(analysis_comp, predictor)
    for dataset in dataset_values:
        dropped_frame = analysis_comp[~analysis_comp["dataset"].eq(dataset)]
        loo_statistic, loo_n = _safe_spearman(dropped_frame, predictor)
        lodo_detail_rows.append(
            {
                "predictor": predictor,
                "dropped_dataset": dataset,
                "statistic": loo_statistic,
                "n_observations": loo_n,
                "abs_change": abs(loo_statistic - full_statistic)
                if pd.notna(loo_statistic) and pd.notna(full_statistic)
                else np.nan,
            }
        )

    predictor_detail = pd.DataFrame([row for row in lodo_detail_rows if row["predictor"] == predictor])
    valid_stats = predictor_detail["statistic"].dropna() if not predictor_detail.empty else pd.Series(dtype=float)
    full_sign = np.sign(full_statistic) if pd.notna(full_statistic) else np.nan
    if valid_stats.empty or pd.isna(full_sign) or full_sign == 0:
        same_sign_share = np.nan
    else:
        same_sign_share = float((np.sign(valid_stats) == full_sign).mean())

    if predictor_detail.empty or predictor_detail["abs_change"].dropna().empty:
        most_influential = None
    else:
        most_influential = predictor_detail.sort_values("abs_change", ascending=False).iloc[0]["dropped_dataset"]

    lodo_rows.append(
        {
            "predictor": predictor,
            "full_statistic": full_statistic,
            "n_full": full_n,
            "loo_min": float(valid_stats.min()) if not valid_stats.empty else np.nan,
            "loo_max": float(valid_stats.max()) if not valid_stats.empty else np.nan,
            "same_sign_share": same_sign_share,
            "most_influential_dropped_dataset": most_influential,
        }
    )

lodo_detail = pd.DataFrame(lodo_detail_rows)
lodo_summary = pd.DataFrame(lodo_rows)
display(lodo_summary)

plot_df = lodo_summary.dropna(subset=["loo_min", "loo_max"])
if plot_df.empty:
    print("No finite leave-one-dataset-out ranges to plot.")
else:
    fig_height = max(3, 0.42 * len(plot_df) + 1.5)
    fig, ax = plt.subplots(figsize=(8, fig_height))
    y_positions = np.arange(len(plot_df))
    ax.hlines(y_positions, plot_df["loo_min"], plot_df["loo_max"], color="0.35", linewidth=2)
    ax.scatter(plot_df["full_statistic"], y_positions, color="black", zorder=3, label="full sample")
    ax.axvline(0, color="0.6", linestyle=":", linewidth=1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_df["predictor"])
    ax.set_xlabel("Spearman rho after dropping one dataset")
    ax.set_title("Leave-one-dataset-out statistic ranges")
    ax.legend(loc="best")
    fig.tight_layout()

## Final Interpretation Aid

This table is a compact checklist for follow-up interpretation. It helps identify patterns that survive simple controls and leave-one-dataset-out checks, but it does not prove causality.

In [ ]:
def _coefficient_sign(value: float) -> str:
    if pd.isna(value):
        return "NA"
    if value > 0:
        return "positive"
    if value < 0:
        return "negative"
    return "zero"


def _redundancy_warning_for(predictor: str) -> str:
    if high_corr_pairs.empty:
        return ""
    matches = high_corr_pairs[
        high_corr_pairs["predictor_1"].eq(predictor) | high_corr_pairs["predictor_2"].eq(predictor)
    ]
    warnings = []
    for row in matches.itertuples(index=False):
        other = row.predictor_2 if row.predictor_1 == predictor else row.predictor_1
        relation = "control" if other in present_controls else "selected"
        warnings.append(f"{other} ({relation}, rho={row.rho:.2f})")
    return "; ".join(warnings)


corr_lookup = corr_comp.set_index("predictor", drop=False)
reg_lookup = regression_results.set_index("predictor", drop=False) if not regression_results.empty else pd.DataFrame()
lodo_lookup = lodo_summary.set_index("predictor", drop=False) if not lodo_summary.empty else pd.DataFrame()

summary_rows = []
for predictor in selected_predictors:
    corr_row = corr_lookup.loc[predictor] if predictor in corr_lookup.index else pd.Series(dtype=object)
    reg_row = reg_lookup.loc[predictor] if predictor in reg_lookup.index else pd.Series(dtype=object)
    lodo_row = lodo_lookup.loc[predictor] if predictor in lodo_lookup.index else pd.Series(dtype=object)
    univariate_p = corr_row[p_sort_col] if p_sort_col in corr_row.index else np.nan
    summary_rows.append(
        {
            "predictor": predictor,
            "univariate_rank": corr_row.get("univariate_rank", np.nan),
            f"univariate_{p_sort_col}": univariate_p,
            "univariate_statistic": corr_row.get("statistic", np.nan),
            "controlled_p_value": reg_row.get("p_value", np.nan),
            "controlled_coefficient_sign": _coefficient_sign(reg_row.get("coefficient", np.nan)),
            "loo_same_sign_share": lodo_row.get("same_sign_share", np.nan),
            "redundancy_warning": _redundancy_warning_for(predictor),
        }
    )

interpretation_summary = pd.DataFrame(summary_rows)
display(interpretation_summary)